In [2]:
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
from gen_variable_standard_static import metrics_search_for_fragment_df, \
metrics_search_for_two_fragments_df, widened_search_for_fragment_df, \
widened_search_for_two_fragments_df, metadata_agg_name, \
metadata_agg_subtype_name, sources_checker, \
find_aggregate_variable_names_gen_mod, \
find_all_variable_names_gen_mod

In [3]:
systems_cleaned = pd.read_csv('../../../data/core/systems_cleaned.csv')

In [7]:
all_data_systems = systems_cleaned[
    systems_cleaned['has_current_data']
    & systems_cleaned['has_irradiance_data']
    & systems_cleaned['has_power_data']
    & systems_cleaned['has_temperature_data']
    & systems_cleaned['has_voltage_data']
]
all_data_ids = set(all_data_systems.system_id)

In [9]:
metrics_dir = Path("../../../data/raw/parquet-metrics/")
metrics_pq = pq.ParquetDataset(
    metrics_dir,
)
metrics_df = metrics_pq.read().to_pandas()
metrics_id_set = set(metrics_df.system_id)

In [10]:
my_system_ids = list(all_data_ids.intersection(metrics_id_set))
my_system_ids.sort()
num_ids = len(my_system_ids)
num_ids

39

## DC Voltage Starter

In [197]:
j = 14
system_id = my_system_ids[j]
relevant_rows_metrics = metrics_df[metrics_df['system_id'] == system_id]
relevant_dc_volts = metrics_search_for_two_fragments_df(
    relevant_rows_metrics, 'dc', 'volt', 'and'
)
relevant_dc_volts

,system_id,metric_id,sensor_name,common_name,raw_units,units,calc_scale,calc_offset,calc_details,aggregation_type,source_type,source_id,comments,standard_name
198,1204,2918,inv1_dc_voltage,DC voltage,V,V,1.0,0.0,,avg,NaN,NaN,,inv1_dc_voltage__2918
199,1204,2922,inv2_dc_voltage,DC voltage,V,V,1.0,0.0,,avg,NaN,NaN,,inv2_dc_voltage__2922
200,1204,2926,inv3_dc_voltage,DC voltage,V,V,1.0,0.0,,avg,NaN,NaN,,inv3_dc_voltage__2926
201,1204,2930,inv4_dc_voltage,DC voltage,V,V,1.0,0.0,,avg,NaN,NaN,,inv4_dc_voltage__2930
202,1204,2934,inv5_dc_voltage,DC voltage,V,V,1.0,0.0,,avg,NaN,NaN,,inv5_dc_voltage__2934
203,1204,2938,inv6_dc_voltage,DC voltage,V,V,1.0,0.0,,avg,NaN,NaN,,inv6_dc_voltage__2938
204,1204,2942,inv7_dc_voltage,DC voltage,V,V,1.0,0.0,,avg,NaN,NaN,,inv7_dc_voltage__2942
205,1204,2946,inv8_dc_voltage,DC voltage,V,V,1.0,0.0,,avg,NaN,NaN,,inv8_dc_voltage__2946
206,1204,2950,inv9_dc_voltage,DC voltage,V,V,1.0,0.0,,avg,NaN,NaN,,inv9_dc_voltage__2950


Note: As from the first-few rich-data systems, often a `dc_pos_voltage` without a `dc_neg_voltage`, just a `das_battery_voltage`.  This makes the agg-vs-parts work very interesting, as System 50 has pos and neg, and no real combination.

*Many* systems have subparts without an aggregate (1200, 1305, 1306,1307,1308,1310, 1403...)  -- more without aggregate than with, I think!

In [ ]:
dc_volts_agg_names = ['dc_voltage', 'input_voltage', 'dc_voltage_Vdc',
                      'InvVDCin_Avg', 'InvPVin_Avg',]  # dc_pos_voltage and dc_neg_voltage removed for now, as 'parts' in a sense
dc_volts_bat_agg_names = ['das_battery_voltage', 'Battery_V_Min']

In [85]:
def find_aggregate_dc_voltage(
    systems_cleaned: pd.DataFrame,
    metrics_df: pd.DataFrame,
    print_messages: bool,
    include_battery: bool,
    known_sources=('inverter', 'battery'),
    known_sources_short=('inv', 'batt')
):
    filtered_metrics_df = metrics_search_for_two_fragments_df(
        metrics_df, 'dc', 'volt', 'and'
    )
    if not include_battery:
        battery_df = metrics_search_for_fragment_df(
            metrics_df, 'batt'
        )
        filtered_metrics_df = filtered_metrics_df.drop(
            index = battery_df.index
        )
    dc_volts_agg_names = ['dc_voltage', 'input_voltage', 'dc_voltage_Vdc',
                          'InvVDCin_Avg', 'InvPVin_Avg',]
    dc_volts_bat_agg_names = ['das_battery_voltage', 'Battery_V_Min']
    all_dc_volts = dc_volts_agg_names + dc_volts_bat_agg_names
    if include_battery:
        return find_aggregate_variable_names_gen_mod(
            systems_cleaned=systems_cleaned,
            filtered_metrics_df=filtered_metrics_df,
            var_name='dc_voltage',
            agg_var_sensor_names=all_dc_volts,
            print_messages=print_messages,
            sources_matter=True,
            known_sources=known_sources,
            known_sources_short=known_sources_short
        )
    else:
        return find_aggregate_variable_names_gen_mod(
            systems_cleaned=systems_cleaned,
            filtered_metrics_df=filtered_metrics_df,
            var_name='dc_voltage',
            agg_var_sensor_names=dc_volts_agg_names,
            print_messages=print_messages,
            sources_matter=False,
            known_sources=known_sources,
            known_sources_short=known_sources_short
        )


In [80]:
all_agg_volts, all_aggvolt_metadata = find_aggregate_dc_voltage(
    systems_cleaned=systems_cleaned,
    metrics_df=metrics_df,
    print_messages=True,
    include_battery=True
)

System 1283 appears to have no obvious dc_voltage aggregator name.
System 1419 appears to have no obvious dc_voltage aggregator name.
System 1422 appears to have no obvious dc_voltage aggregator name.
System 1423 appears to have no obvious dc_voltage aggregator name.
System 1429 appears to have no obvious dc_voltage aggregator name.
System 1305 appears to have no obvious dc_voltage aggregator name.
System 1306 appears to have no obvious dc_voltage aggregator name.
System 1307 appears to have no obvious dc_voltage aggregator name.
System 1308 appears to have no obvious dc_voltage aggregator name.
System 1310 appears to have no obvious dc_voltage aggregator name.
System 1312 appears to have no obvious dc_voltage aggregator name.
System 1199 appears to have no obvious dc_voltage aggregator name.
System 1200 appears to have no obvious dc_voltage aggregator name.
System 1202 appears to have no obvious dc_voltage aggregator name.
System 1204 appears to have no obvious dc_voltage aggregator n

In [ ]:
def find_all_dc_voltage(
    systems_cleaned: pd.DataFrame,
    metrics_df: pd.DataFrame,
    print_messages: bool,
    include_battery: bool,
    known_sources=('inverter', 'battery'),
    known_sources_short=('inv', 'batt')
):
    filtered_metrics_df = metrics_search_for_two_fragments_df(
        metrics_df, 'dc', 'volt', 'and'
    )
    if not include_battery:
        battery_df = metrics_search_for_fragment_df(
            metrics_df, 'batt'
        )
        filtered_metrics_df = filtered_metrics_df.drop(
            index = battery_df.index
        )
    dc_volts_agg_names = ['dc_voltage', 'input_voltage', 'dc_voltage_Vdc',
                          'InvVDCin_Avg', 'InvPVin_Avg',]
    dc_volts_bat_agg_names = ['das_battery_voltage', 'Battery_V_Min']
    all_dc_volts = dc_volts_agg_names + dc_volts_bat_agg_names
    all_agg_volts, all_aggvolt_metadata = find_aggregate_dc_voltage(
        systems_cleaned=systems_cleaned,
        metrics_df=metrics_df,
        print_messages=print_messages,
        include_battery=include_battery,
        known_sources=known_sources,
        known_sources_short=known_sources_short
    )
    if include_battery:
        return find_all_variable_names_gen_mod(
            var_aggs_dict=all_agg_volts,
            var_aggs_metadata=all_aggvolt_metadata,
            filtered_metrics_df=filtered_metrics_df,
            var_name='dc_voltage',
            agg_var_sensor_names=all_dc_volts,
            sources_matter=True,
            known_sources=known_sources,
            known_sources_short=known_sources_short
        )
    else:
        return find_all_variable_names_gen_mod(
            var_aggs_dict=all_agg_volts,
            var_aggs_metadata=all_aggvolt_metadata,
            filtered_metrics_df=filtered_metrics_df,
            var_name='dc_voltage',
            agg_var_sensor_names=all_dc_volts,
            sources_matter=False,
            known_sources=known_sources,
            known_sources_short=known_sources_short
        )

In [89]:
all_volts, all_metadata = find_all_dc_voltage(
    systems_cleaned=systems_cleaned,
    metrics_df=metrics_df,
    print_messages=False,
    include_battery=True
)

System 2 has only one subpart for dc_voltage!
system_id                             2
metric_id                           347
sensor_name              dc_pos_voltage
common_name                  DC voltage
raw_units                             V
units                                 V
calc_scale                          1.0
calc_offset                         0.0
calc_details                           
aggregation_type                    avg
source_type                         NaN
source_id                           NaN
comments                               
standard_name       dc_pos_voltage__347
Name: 1302, dtype: object


ValueError: Incorrect subpart description, presumably.

Oof!  Cannot fix this without dropping the errors!

## AC Voltage Starter

In [199]:
# j = 15
# system_id = my_system_ids[j]
system_id = 1214
relevant_rows_metrics = metrics_df[metrics_df['system_id'] == system_id]
relevant_ac_volts = metrics_search_for_two_fragments_df(
    relevant_rows_metrics, 'ac', 'volt', 'and'
)
relevant_ac_volts

,system_id,metric_id,sensor_name,common_name,raw_units,units,calc_scale,calc_offset,calc_details,aggregation_type,source_type,source_id,comments,standard_name
240,1214,3184,PPV_avg,AC voltage,V,V,1.0,0.0,,avg,NaN,NaN,,ppv_avg__3184


Immediate observation: battery or no battery, sources matter!
Still lots of missingness for aggregates!

In [138]:
ac_voltage_agg_names = [
    'InvVabcAvg_Avg', 'ac_voltage_Vac',
    'ac_voltage', 'ac_volts',
]

In [200]:
def find_aggregate_ac_voltage(
    systems_cleaned: pd.DataFrame,
    metrics_df: pd.DataFrame,
    print_messages: bool,
    known_sources=('inverter', 'meter'),
    known_sources_short=('inv', 'met')
):
    filtered_metrics_df = metrics_search_for_two_fragments_df(
        metrics_df, 'ac', 'volt', 'and'
    )
    ac_voltage_agg_names = [
        'InvVabcAvg_Avg', 'ac_voltage_Vac',
        'ac_voltage', 'ac_volts', 'PPV_avg'
    ]
    return find_aggregate_variable_names_gen_mod(
        systems_cleaned=systems_cleaned,
        filtered_metrics_df=filtered_metrics_df,
        var_name='dc_voltage',
        agg_var_sensor_names=ac_voltage_agg_names,
        print_messages=print_messages,
        sources_matter=True,
        known_sources=known_sources,
        known_sources_short=known_sources_short
    )


In [156]:
all_agg_ac_voltage, all_agg_ac_voltage_metadata = find_aggregate_ac_voltage(
    systems_cleaned=systems_cleaned,
    metrics_df=metrics_df,
    print_messages=True
)

System 1416 appears to have no obvious dc_voltage aggregator name.
System 1422 appears to have no obvious dc_voltage aggregator name.
System 1423 appears to have no obvious dc_voltage aggregator name.
System 1429 appears to have no obvious dc_voltage aggregator name.
System 1305 appears to have no obvious dc_voltage aggregator name.
System 1306 appears to have no obvious dc_voltage aggregator name.
System 1307 appears to have no obvious dc_voltage aggregator name.
System 1308 appears to have no obvious dc_voltage aggregator name.
System 1310 appears to have no obvious dc_voltage aggregator name.
System 1312 appears to have no obvious dc_voltage aggregator name.
System 4901 appears to have no obvious dc_voltage aggregator name.
System 4902 appears to have no obvious dc_voltage aggregator name.
System 1203 appears to have no obvious dc_voltage aggregator name.
System 1368 appears to have no obvious dc_voltage aggregator name.
System 1369 appears to have no obvious dc_voltage aggregator n

In [201]:
def find_all_ac_voltage(
    systems_cleaned: pd.DataFrame,
    metrics_df: pd.DataFrame,
    print_messages: bool,
    known_sources=('inverter', 'meter'),
    known_sources_short=('inv', 'met')
):
    filtered_metrics_df = metrics_search_for_two_fragments_df(
        metrics_df, 'ac', 'volt', 'and'
    )
    # no percentages
    filtered_metrics_df = filtered_metrics_df[filtered_metrics_df['units'] == 'V']
    ac_voltage_agg_names = [
        'InvVabcAvg_Avg', 'ac_voltage_Vac',
        'ac_voltage', 'ac_volts', 'PPV_avg'
    ]
    all_agg_ac_volts, all_agg_ac_volt_metadata = find_aggregate_ac_voltage(
        systems_cleaned=systems_cleaned,
        metrics_df=metrics_df,
        print_messages=print_messages,
        known_sources=known_sources,
        known_sources_short=known_sources_short
    )
    return find_all_variable_names_gen_mod(
        var_aggs_dict=all_agg_ac_volts,
        var_aggs_metadata=all_agg_ac_volt_metadata,
        filtered_metrics_df=filtered_metrics_df,
        var_name='ac_voltage',
        agg_var_sensor_names=ac_voltage_agg_names,
        sources_matter=True,
        known_sources=known_sources,
        known_sources_short=known_sources_short
    )


In [202]:
all_ac_volts, all_ac_metadata = find_all_ac_voltage(
    systems_cleaned=systems_cleaned,
    metrics_df=metrics_df,
    print_messages=False,
)

In [203]:
units_list = []
for system_id in all_ac_volts.keys():
    for metric_dict in all_ac_volts[system_id]:
        units_list.append(metric_dict['units'])
units_list = set(units_list)
units_list
    

{'V'}